In [1]:
import urllib.request
import os
import datetime

def get_vhi(pid, y1=1981, y2=2024, fld="vhi_data"):
    if not os.path.exists(fld):
        os.makedirs(fld)
        
    ex = [f for f in os.listdir(fld) if f.startswith(f"vhi_id_{pid}_")]
    if ex:
        print("Exists")
        return os.path.join(fld, ex[0])
    
    url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={pid}&year1={y1}&year2={y2}&type=Mean"
    
    now = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    fname = f"vhi_id_{pid}_{now}.csv"
    fpath = os.path.join(fld, fname)
    
    try:
        urllib.request.urlretrieve(url, fpath)
        print("Downloaded")
        return fpath
    except Exception:
        print("Error")
        return None

print("Start")
for i in range(1, 28):
    get_vhi(i)
print("Done")

Start
Downloaded
Downloaded
Downloaded
Downloaded
Downloaded
Downloaded
Downloaded
Downloaded
Downloaded
Downloaded
Downloaded
Downloaded
Downloaded
Downloaded
Downloaded
Downloaded
Downloaded
Downloaded
Downloaded
Downloaded
Downloaded
Downloaded
Downloaded
Downloaded
Downloaded
Downloaded
Downloaded
Done


In [2]:
import pandas as pd
import os

def bld_df(fld="vhi_data"):
    p_map = {1: 22, 2: 24, 3: 23, 4: 25, 5: 3, 6: 4, 7: 8, 8: 19, 9: 20, 10: 21, 11: 9, 12: 26, 13: 10, 14: 11, 15: 12, 16: 13, 17: 14, 18: 15, 19: 16, 20: 27, 21: 17, 22: 18, 23: 6, 24: 1, 25: 2, 26: 7, 27: 5}
    dfs = []
    
    for f in os.listdir(fld):
        if not f.endswith('.csv'): 
            continue
        pid = int(f.split('_')[2])
        
        d = pd.read_csv(os.path.join(fld, f), header=1, names=['year', 'week', 'smn', 'smt', 'vci', 'tci', 'vhi', 'empty'])
        d.drop(columns=['empty'], inplace=True, errors='ignore')
        d.dropna(inplace=True)
        
        d['year'] = d['year'].astype(str).str.replace(r'<.*?>', '', regex=True)
        d = d[d['year'].str.strip() != '']
        d['year'] = d['year'].astype(int)
        
        d = d[d['vhi'] != -1.0]
        d['prv'] = p_map.get(pid, pid)
        dfs.append(d)
        
    res = pd.concat(dfs).reset_index(drop=True)
    print("Cleaned")
    return res
    
df = bld_df()

Cleaned


In [3]:
def f_y(d, p, y):
    res = d[(d['prv'] == p) & (d['year'] == y)]
    print("Filtered")
    return res

def f_ry(d, p_ls, y1, y2):
    res = d[(d['prv'].isin(p_ls)) & (d['year'] >= y1) & (d['year'] <= y2)]
    print("Filtered")
    return res

def f_ext(d, p_ls, y1, y2):
    sub = d[(d['prv'].isin(p_ls)) & (d['year'] >= y1) & (d['year'] <= y2)]
    res = {
        'min': sub['vhi'].min(),
        'max': sub['vhi'].max(),
        'mean': sub['vhi'].mean(),
        'med': sub['vhi'].median()
    }
    print("Calculated")
    return res

v1 = f_y(df, 1, 2020)
v2 = f_ry(df, [1, 2], 2010, 2020)
v3 = f_ext(df, [1, 2], 2010, 2020)

Filtered
Filtered
Calculated
